In [1]:
!pip install tinycss2 -q

In [7]:
from pathlib import Path
from collections import Counter, defaultdict
import tinycss2

In [8]:
def analyze_css(css_path):
    css_path = Path(css_path)

    if not css_path.exists():
        raise FileNotFoundError(f"CSS file not found: {css_path}")

    css_text = css_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

    stylesheet = tinycss2.parse_stylesheet(
        css_text,
        skip_comments=True,
        skip_whitespace=True
    )

    rules_inventory = []
    selector_counts = Counter()
    property_counts = Counter()

    source_order = 0


    def split_selectors(tokens):
        """
        Split selector list by top-level commas.
        Safer than string.split(',') for selectors containing functions.
        """
        selectors = []
        current = []

        for token in tokens:
            if token.type == "literal" and token.value == ",":
                selector = tinycss2.serialize(current).strip()

                if selector:
                    selectors.append(selector)

                current = []
            else:
                current.append(token)

        selector = tinycss2.serialize(current).strip()

        if selector:
            selectors.append(selector)

        return selectors


    def parse_declarations(content):
        declarations = tinycss2.parse_declaration_list(
            content,
            skip_comments=True,
            skip_whitespace=True
        )

        result = []

        for declaration in declarations:

            if declaration.type != "declaration":
                continue

            property_name = declaration.lower_name

            value = tinycss2.serialize(
                declaration.value
            ).strip()

            result.append({
                "property": property_name,
                "value": value,
                "important": declaration.important
            })

            property_counts[property_name] += 1

        return result


    def process_rules(rule_list, context=()):
        nonlocal source_order

        for rule in rule_list:

            # -----------------------------
            # Normal CSS rule
            # -----------------------------
            if rule.type == "qualified-rule":

                selectors = split_selectors(rule.prelude)
                declarations = parse_declarations(rule.content)

                for selector in selectors:

                    selector_counts[selector] += 1

                    rules_inventory.append({
                        "order": source_order,
                        "selector": selector,
                        "context": context,
                        "declarations": declarations.copy()
                    })

                    source_order += 1


            # -----------------------------
            # Nested @ rules
            # -----------------------------
            elif rule.type == "at-rule":

                at_name = rule.lower_at_keyword

                prelude = tinycss2.serialize(
                    rule.prelude
                ).strip()

                current_context = context + (
                    f"@{at_name} {prelude}".strip(),
                )

                if rule.content is None:
                    continue

                # These normally contain CSS rules
                if at_name in {
                    "media",
                    "supports",
                    "layer",
                    "container",
                    "scope",
                    "document"
                }:

                    nested_rules = tinycss2.parse_rule_list(
                        rule.content,
                        skip_comments=True,
                        skip_whitespace=True
                    )

                    process_rules(
                        nested_rules,
                        current_context
                    )


    process_rules(stylesheet)


    # ---------------------------------------
    # Find exact duplicate rules
    # ---------------------------------------

    duplicate_groups = defaultdict(list)

    for rule in rules_inventory:

        declaration_signature = tuple(
            (
                d["property"],
                d["value"],
                d["important"]
            )
            for d in rule["declarations"]
        )

        signature = (
            rule["context"],
            rule["selector"],
            declaration_signature
        )

        duplicate_groups[signature].append(rule["order"])


    exact_duplicates = []

    for signature, orders in duplicate_groups.items():

        if len(orders) <= 1:
            continue

        context, selector, declarations = signature

        exact_duplicates.append({
            "selector": selector,
            "context": context,
            "occurrences": len(orders),
            "orders": orders,
            "removable": len(orders) - 1,
            "declarations": declarations
        })


    return {
        "file": str(css_path),

        "total_rule_instances": len(rules_inventory),
        "unique_selectors": len(selector_counts),
        "unique_properties": len(property_counts),

        "selector_counts": selector_counts,
        "property_counts": property_counts,

        "rules": rules_inventory,
        "exact_duplicates": exact_duplicates
    }

In [9]:
CSS_PATH = r"/mnt/e/Desk/IOTAML/iota_ml/frontend/src/styles/00-foundation-workflow.css"

result = analyze_css(CSS_PATH)

print("Total rules:", result["total_rule_instances"])
print("Unique selectors:", result["unique_selectors"])
print("Unique properties:", result["unique_properties"])
print("Exact duplicate groups:", len(result["exact_duplicates"]))

Total rules: 647
Unique selectors: 340
Unique properties: 96
Exact duplicate groups: 27


In [10]:
for dup in result["exact_duplicates"]:
    print("\nSelector:", dup["selector"])
    print("Context:", dup["context"] or "ROOT")
    print("Occurrences:", dup["occurrences"])
    print("Can remove:", dup["removable"])
    print("Source orders:", dup["orders"])


Selector: .palette
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [43, 464]

Selector: .inspector
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [44, 465]

Selector: .results-panel
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [45, 466]

Selector: .dataset-card
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [46, 467]

Selector: .flow-chip
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [47, 468]

Selector: .palette
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [186, 255]

Selector: .inspector
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [187, 256]

Selector: .results-panel
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [188, 257]

Selector: .dataset-card
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [189, 258]

Selector: .flow-chip
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [190, 259]

Selector: .board
Context: ROOT
Occurrences: 2
Can remove: 1
Source orders: [191, 476

In [11]:
from collections import defaultdict


def find_property_conflicts(analysis):
    """
    Detect repeated properties for the same selector in the same context.

    Detects:
      - duplicate_same_value
      - overridden
      - important_override
      - shadowed_by_important

    Does NOT modify CSS.
    """

    groups = defaultdict(list)

    # --------------------------------------------------
    # Collect every declaration with its source position
    # --------------------------------------------------
    for rule in analysis["rules"]:

        key = (
            rule["context"],
            rule["selector"]
        )

        for declaration_index, declaration in enumerate(rule["declarations"]):

            groups[key].append({
                "order": rule["order"],
                "declaration_index": declaration_index,
                "property": declaration["property"],
                "value": declaration["value"],
                "important": declaration["important"]
            })

    conflicts = []

    # --------------------------------------------------
    # Analyze each selector/context independently
    # --------------------------------------------------
    for (context, selector), declarations in groups.items():

        by_property = defaultdict(list)

        for declaration in declarations:
            by_property[declaration["property"]].append(declaration)

        property_conflicts = []

        for property_name, occurrences in by_property.items():

            if len(occurrences) < 2:
                continue

            # Sort exactly by CSS source order
            occurrences.sort(
                key=lambda x: (
                    x["order"],
                    x["declaration_index"]
                )
            )

            # ------------------------------------------
            # Determine winning declaration
            #
            # !important beats normal.
            # Otherwise later declaration wins.
            # ------------------------------------------
            important_occurrences = [
                x for x in occurrences
                if x["important"]
            ]

            if important_occurrences:
                winner = important_occurrences[-1]
            else:
                winner = occurrences[-1]

            occurrence_details = []

            for occurrence in occurrences:

                is_winner = occurrence is winner

                if is_winner:
                    status = "winner"

                elif (
                    occurrence["value"] == winner["value"]
                    and occurrence["important"] == winner["important"]
                ):
                    status = "duplicate_same_value"

                elif winner["important"] and not occurrence["important"]:
                    status = "shadowed_by_important"

                elif occurrence["important"] and winner["important"]:
                    status = "overridden"

                else:
                    status = "overridden"

                occurrence_details.append({
                    **occurrence,
                    "status": status
                })

            # Classify overall conflict
            unique_values = {
                (x["value"], x["important"])
                for x in occurrences
            }

            if len(unique_values) == 1:
                conflict_type = "duplicate_same_value"

            elif any(x["important"] for x in occurrences):
                conflict_type = "important_conflict"

            else:
                conflict_type = "value_override"

            property_conflicts.append({
                "property": property_name,
                "type": conflict_type,
                "count": len(occurrences),
                "winner": winner,
                "occurrences": occurrence_details
            })

        if property_conflicts:
            conflicts.append({
                "selector": selector,
                "context": context,
                "properties": property_conflicts
            })

    return conflicts

In [12]:
conflicts = find_property_conflicts(result)

print("Selectors with repeated properties:", len(conflicts))

Selectors with repeated properties: 87


In [13]:
for item in conflicts:

    print("\n" + "=" * 70)
    print("SELECTOR:", item["selector"])
    print("CONTEXT :", item["context"] or "ROOT")

    for prop in item["properties"]:

        print(f"\n  {prop['property']} [{prop['type']}]")

        for occurrence in prop["occurrences"]:

            important = " !important" if occurrence["important"] else ""

            print(
                f"    rule #{occurrence['order']}: "
                f"{occurrence['value']}{important}"
                f"  -> {occurrence['status']}"
            )


SELECTOR: :root
CONTEXT : ROOT

  color-scheme [duplicate_same_value]
    rule #0: dark  -> duplicate_same_value
    rule #451: dark  -> winner

  background [duplicate_same_value]
    rule #0: var(--theme-app-bg)  -> duplicate_same_value
    rule #451: var(--theme-app-bg)  -> winner

  color [duplicate_same_value]
    rule #0: var(--theme-text)  -> duplicate_same_value
    rule #451: var(--theme-text)  -> winner

SELECTOR: body
CONTEXT : ROOT

  min-width [duplicate_same_value]
    rule #250: 0  -> duplicate_same_value
    rule #251: 0  -> winner

SELECTOR: button
CONTEXT : ROOT

  border-radius [duplicate_same_value]
    rule #198: var(--theme-radius-control)  -> duplicate_same_value
    rule #262: var(--theme-radius-control)  -> winner

SELECTOR: input
CONTEXT : ROOT

  border-radius [duplicate_same_value]
    rule #199: var(--theme-radius-control)  -> duplicate_same_value
    rule #263: var(--theme-radius-control)  -> winner

SELECTOR: .topbar
CONTEXT : ROOT

  position [duplicate

In [14]:
from collections import defaultdict


# Shorthand -> properties it can affect
SHORTHAND_MAP = {
    "margin": {
        "margin-top",
        "margin-right",
        "margin-bottom",
        "margin-left",
    },

    "padding": {
        "padding-top",
        "padding-right",
        "padding-bottom",
        "padding-left",
    },

    "inset": {
        "top",
        "right",
        "bottom",
        "left",
    },

    "border-width": {
        "border-top-width",
        "border-right-width",
        "border-bottom-width",
        "border-left-width",
    },

    "border-style": {
        "border-top-style",
        "border-right-style",
        "border-bottom-style",
        "border-left-style",
    },

    "border-color": {
        "border-top-color",
        "border-right-color",
        "border-bottom-color",
        "border-left-color",
    },

    "border": {
        "border-top-width",
        "border-right-width",
        "border-bottom-width",
        "border-left-width",

        "border-top-style",
        "border-right-style",
        "border-bottom-style",
        "border-left-style",

        "border-top-color",
        "border-right-color",
        "border-bottom-color",
        "border-left-color",
    },

    "border-top": {
        "border-top-width",
        "border-top-style",
        "border-top-color",
    },

    "border-right": {
        "border-right-width",
        "border-right-style",
        "border-right-color",
    },

    "border-bottom": {
        "border-bottom-width",
        "border-bottom-style",
        "border-bottom-color",
    },

    "border-left": {
        "border-left-width",
        "border-left-style",
        "border-left-color",
    },

    "flex": {
        "flex-grow",
        "flex-shrink",
        "flex-basis",
    },

    "gap": {
        "row-gap",
        "column-gap",
    },

    "overflow": {
        "overflow-x",
        "overflow-y",
    },

    "columns": {
        "column-width",
        "column-count",
    },

    "list-style": {
        "list-style-type",
        "list-style-position",
        "list-style-image",
    },
}


def find_shorthand_conflicts(analysis):
    """
    Find declarations where shorthand and longhand properties
    can affect the same final CSS property.

    Important:
    This is a conservative detector.
    It REPORTS conflicts but does not remove anything.
    """

    groups = defaultdict(list)

    # ------------------------------------------------
    # Collect declarations by selector + context
    # ------------------------------------------------
    for rule in analysis["rules"]:

        key = (
            rule["context"],
            rule["selector"]
        )

        for declaration_index, declaration in enumerate(
            rule["declarations"]
        ):

            groups[key].append({
                "order": rule["order"],
                "declaration_index": declaration_index,
                "property": declaration["property"],
                "value": declaration["value"],
                "important": declaration["important"],
            })

    results = []

    # ------------------------------------------------
    # Process each selector independently
    # ------------------------------------------------
    for (context, selector), declarations in groups.items():

        declarations.sort(
            key=lambda x: (
                x["order"],
                x["declaration_index"]
            )
        )

        conflicts = []

        # Compare each declaration with declarations after it
        for i, first in enumerate(declarations):

            for second in declarations[i + 1:]:

                prop1 = first["property"]
                prop2 = second["property"]

                affected1 = SHORTHAND_MAP.get(prop1)
                affected2 = SHORTHAND_MAP.get(prop2)

                conflict_kind = None

                # ----------------------------------------
                # shorthand -> longhand
                # margin -> margin-left
                # ----------------------------------------
                if affected1 and prop2 in affected1:
                    conflict_kind = "shorthand_then_longhand"

                # ----------------------------------------
                # longhand -> shorthand
                # margin-left -> margin
                # ----------------------------------------
                elif affected2 and prop1 in affected2:
                    conflict_kind = "longhand_then_shorthand"

                # ----------------------------------------
                # overlapping shorthands
                # border -> border-left
                # ----------------------------------------
                elif affected1 and affected2:

                    overlap = affected1.intersection(affected2)

                    if overlap:
                        conflict_kind = "overlapping_shorthands"

                if not conflict_kind:
                    continue

                # ----------------------------------------
                # Basic importance relationship
                # ----------------------------------------
                if first["important"] and not second["important"]:
                    likely_effect = "earlier_important_may_win"

                elif second["important"] and not first["important"]:
                    likely_effect = "later_important_wins"

                elif first["important"] == second["important"]:
                    likely_effect = "later_declaration_affects_previous"

                else:
                    likely_effect = "needs_review"

                conflicts.append({
                    "first": first,
                    "second": second,
                    "type": conflict_kind,
                    "effect": likely_effect,
                })

        if conflicts:

            results.append({
                "selector": selector,
                "context": context,
                "conflicts": conflicts,
            })

    return results

In [15]:
shorthand_conflicts = find_shorthand_conflicts(result)

print(
    "Selectors with shorthand/longhand conflicts:",
    len(shorthand_conflicts)
)

Selectors with shorthand/longhand conflicts: 51


In [16]:
for item in shorthand_conflicts:

    print("\n" + "=" * 75)
    print("SELECTOR:", item["selector"])
    print("CONTEXT :", item["context"] or "ROOT")

    for conflict in item["conflicts"]:

        a = conflict["first"]
        b = conflict["second"]

        imp_a = " !important" if a["important"] else ""
        imp_b = " !important" if b["important"] else ""

        print(
            f"\n  {a['property']}: {a['value']}{imp_a}"
        )

        print(
            f"  -> {b['property']}: {b['value']}{imp_b}"
        )

        print(
            f"  Type: {conflict['type']}"
        )

        print(
            f"  Effect: {conflict['effect']}"
        )


SELECTOR: .run-controls input
CONTEXT : ROOT

  border: 0 solid var(--theme-control-border)
  -> border-color: color-mix(in srgb,var(--theme-control-border) 80%,var(--theme-primary))
  Type: overlapping_shorthands
  Effect: later_declaration_affects_previous

  border: 0 solid var(--theme-control-border)
  -> border: 0px solid var(--theme-divider)
  Type: overlapping_shorthands
  Effect: later_declaration_affects_previous

  border-color: color-mix(in srgb,var(--theme-control-border) 80%,var(--theme-primary))
  -> border: 0px solid var(--theme-divider)
  Type: overlapping_shorthands
  Effect: later_declaration_affects_previous

SELECTOR: .field input
CONTEXT : ROOT

  border: 0 solid var(--theme-control-border)
  -> border-color: color-mix(in srgb,var(--theme-control-border) 80%,var(--theme-primary))
  Type: overlapping_shorthands
  Effect: later_declaration_affects_previous

  border: 0 solid var(--theme-control-border)
  -> border: 0px solid var(--theme-divider)
  Type: overlapping_

In [17]:
from collections import defaultdict


def classify_optimization_candidates(analysis):
    """
    Conservative optimization classifier.

    SAFE:
        Exact duplicate declarations inside the SAME selector rule.

    REVIEW:
        Same property appears multiple times with different values.
        Could intentionally be a browser fallback.

    UNSAFE:
        Similar/repeated declarations across separate rule blocks.
        Requires full cascade analysis before removal.
    """

    candidates = []

    # ============================================================
    # 1. Analyze declarations INSIDE each individual rule
    # ============================================================
    for rule in analysis["rules"]:

        declarations = rule["declarations"]

        # ----------------------------------------
        # Exact duplicates inside same rule
        # ----------------------------------------
        seen = defaultdict(list)

        for index, d in enumerate(declarations):

            signature = (
                d["property"],
                d["value"],
                d["important"]
            )

            seen[signature].append(index)

        for signature, indexes in seen.items():

            if len(indexes) <= 1:
                continue

            prop, value, important = signature

            # Keep last occurrence
            for index in indexes[:-1]:

                candidates.append({
                    "safety": "SAFE",
                    "reason": "exact_duplicate_in_same_rule",

                    "selector": rule["selector"],
                    "context": rule["context"],

                    "rule_order": rule["order"],
                    "declaration_index": index,

                    "property": prop,
                    "value": value,
                    "important": important,

                    "action": "remove"
                })

        # ----------------------------------------
        # Different values for same property
        # ----------------------------------------
        property_groups = defaultdict(list)

        for index, d in enumerate(declarations):

            property_groups[d["property"]].append({
                "index": index,
                **d
            })

        for prop, occurrences in property_groups.items():

            if len(occurrences) <= 1:
                continue

            unique_values = {
                (
                    x["value"],
                    x["important"]
                )
                for x in occurrences
            }

            if len(unique_values) > 1:

                candidates.append({
                    "safety": "REVIEW",
                    "reason": "multiple_values_same_property",

                    "selector": rule["selector"],
                    "context": rule["context"],

                    "rule_order": rule["order"],

                    "property": prop,

                    "occurrences": occurrences,

                    "action": "do_not_remove_yet"
                })

    # ============================================================
    # 2. Cross-rule repeated declarations
    # ============================================================

    cross_rule = defaultdict(list)

    for rule in analysis["rules"]:

        for index, d in enumerate(rule["declarations"]):

            key = (
                rule["context"],
                rule["selector"],
                d["property"]
            )

            cross_rule[key].append({
                "rule_order": rule["order"],
                "declaration_index": index,
                "value": d["value"],
                "important": d["important"]
            })

    for (context, selector, prop), occurrences in cross_rule.items():

        rule_orders = {
            x["rule_order"]
            for x in occurrences
        }

        # Only interested in declarations spanning
        # more than one rule
        if len(rule_orders) <= 1:
            continue

        candidates.append({
            "safety": "UNSAFE",
            "reason": "cross_rule_cascade_dependency",

            "selector": selector,
            "context": context,

            "property": prop,
            "occurrences": occurrences,

            "action": "needs_cascade_analysis"
        })

    return candidates

In [18]:
candidates = classify_optimization_candidates(result)

safe = [x for x in candidates if x["safety"] == "SAFE"]
review = [x for x in candidates if x["safety"] == "REVIEW"]
unsafe = [x for x in candidates if x["safety"] == "UNSAFE"]

print("SAFE removals :", len(safe))
print("REVIEW        :", len(review))
print("UNSAFE        :", len(unsafe))

SAFE removals : 0
REVIEW        : 0
UNSAFE        : 198


In [19]:
for item in safe:
    important = " !important" if item["important"] else ""

    print(
        f'{item["selector"]}  |  '
        f'{item["property"]}: {item["value"]}{important}'
    )

In [20]:
print("Total candidates:", len(candidates))
print("SAFE   :", len(safe))
print("REVIEW :", len(review))
print("UNSAFE :", len(unsafe))

print("\nFirst 20 cross-rule cases:\n")

for item in unsafe[:20]:
    print(
        item["selector"],
        "|",
        item["property"],
        "| occurrences:",
        len(item["occurrences"])
    )

Total candidates: 198
SAFE   : 0
REVIEW : 0
UNSAFE : 198

First 20 cross-rule cases:

:root | color-scheme | occurrences: 2
:root | background | occurrences: 2
:root | color | occurrences: 2
.run-controls input | background | occurrences: 2
.run-controls input | border | occurrences: 2
.field input | background | occurrences: 2
.field input | border | occurrences: 2
.field input | min-width | occurrences: 2
.field select | background | occurrences: 2
.field select | border | occurrences: 2
.field select | min-width | occurrences: 2
.searchbox input | background | occurrences: 2
.searchbox input | border | occurrences: 2
.searchbox input | min-width | occurrences: 2
.primary | font-weight | occurrences: 2
.danger | font-weight | occurrences: 2
.icon-button | font-weight | occurrences: 2
.tiny-action | padding | occurrences: 2
.tiny-action | font-weight | occurrences: 2
.tiny-action | font-size | occurrences: 2
